# XLZD MF-GP Workflow

This notebook assumes the CNP stage has already been run and that the aggregated CNP CSVs exist under `data/out/cnp`.

It covers only the MF-GP part of the XLZD RESuM workflow:

1. load the XLZD MF-GP settings
2. fit the MF-GP using the CNP output from training LF + training HF
3. generate grid predictions and validation plots
4. inspect the saved metrics, CSV outputs, and plots inline


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "prepare_resum_data.py").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not find the XLZD repo root from the current working directory.")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)

if str(REPO_ROOT / "src" / "run_mfgp") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src" / "run_mfgp"))

from mfgp_clean_pipeline import load_runtime_config, run_mfgp_transform_suite

CONFIG_PATH = REPO_ROOT / "src" / "xlzd" / "settings.yaml"
CNP_TRAIN_CSV = REPO_ROOT / "data" / "out" / "cnp" / "cnp_xlzd_v1_output_15epochs.csv"
CNP_VALIDATION_CSV = REPO_ROOT / "data" / "out" / "cnp" / "cnp_xlzd_v1_output_validation_15epochs.csv"
ITERATION = 0
GRID_POINTS = 120
PREDICT_CHUNK_SIZE = 20000
RANDOM_STATE = 42
TARGET_TRANSFORMS = ["linear", "log_hf", "log_lf", "log_both"]

print(f"Repo root: {REPO_ROOT}")
print(f"MF-GP config: {CONFIG_PATH}")
print(f"Training CNP CSV: {CNP_TRAIN_CSV}")
print(f"Validation CNP CSV: {CNP_VALIDATION_CSV}")
print(f"Target transforms: {TARGET_TRANSFORMS}")


## 1. Load And Inspect The Runtime Config

This shows the theta headers and output folders the MF-GP notebook will use.


In [ ]:
runtime = load_runtime_config(CONFIG_PATH)

summary = pd.DataFrame(
    {
        "field": [
            "version",
            "theta_headers",
            "theta_min",
            "theta_max",
            "out_dir_cnp",
            "out_dir_mfgp",
        ],
        "value": [
            runtime.version,
            ", ".join(runtime.theta_headers),
            runtime.theta_min,
            runtime.theta_max,
            str(runtime.out_dir_cnp),
            str(runtime.out_dir_mfgp),
        ],
    }
)
summary


## 2. Fit The MF-GP Transform Suite

This runs four MF-GP variants from the same CNP CSVs: linear baseline, log-HF, log-LF, and log-both.


In [ ]:
mfgp_results = run_mfgp_transform_suite(
    config_path=CONFIG_PATH,
    cnp_csv=CNP_TRAIN_CSV,
    validation_csv=CNP_VALIDATION_CSV,
    transforms=TARGET_TRANSFORMS,
    iteration=ITERATION,
    grid_points_per_axis=GRID_POINTS,
    random_state=RANDOM_STATE,
    predict_chunk_size=PREDICT_CHUNK_SIZE,
    verbose=True,
)

EXPERIMENT_TITLES = {
    "linear": "Normal MF-GP",
    "log_hf": "Log HF: emulate log10(y_raw)",
    "log_lf": "Log LF: use log10(y_cnp)",
    "log_both": "Log HF + Log LF: use log10(y_raw) and log10(y_cnp)",
}

PLOT_SELECTIONS = {
    "linear": [
        ("4-fold MF-GP mean/std", "mean_std_plot"),
        ("Interactive 3D mean/std HTML", "mean_std_plot_3d_html"),
        ("Validation 3-sigma: y with linear sigma", "validation_across_theta_linear_plot"),
        ("Validation 3-sigma: log10(y) with linear sigma", "validation_across_theta_log_linear_sigma_plot"),
        ("Validation parity", "validation_parity_linear_plot"),
    ],
    "log_hf": [
        ("4-fold MF-GP mean/std", "mean_std_plot"),
        ("Interactive 3D mean/std HTML", "mean_std_plot_3d_html"),
        ("Validation 3-sigma: log10 target with linear sigma", "validation_across_theta_linear_plot"),
        ("Validation parity", "validation_parity_linear_plot"),
    ],
    "log_lf": [
        ("4-fold MF-GP mean/std", "mean_std_plot"),
        ("Interactive 3D mean/std HTML", "mean_std_plot_3d_html"),
        ("Validation 3-sigma: y with linear sigma", "validation_across_theta_linear_plot"),
        ("Validation 3-sigma: log10(y) with linear sigma", "validation_across_theta_log_linear_sigma_plot"),
        ("Validation parity", "validation_parity_linear_plot"),
    ],
    "log_both": [
        ("4-fold MF-GP mean/std", "mean_std_plot"),
        ("Interactive 3D mean/std HTML", "mean_std_plot_3d_html"),
        ("Validation 3-sigma: log10 target with linear sigma", "validation_across_theta_linear_plot"),
        ("Validation parity", "validation_parity_linear_plot"),
    ],
}

artifact_rows = []
for mode, result in mfgp_results.items():
    artifact_rows.extend(
        [
            {"mode": mode, "artifact": "cnp_csv", "path": str(result.cnp_csv)},
            {"mode": mode, "artifact": "model_json", "path": str(result.model_json)},
            {"mode": mode, "artifact": "metrics_json", "path": str(result.metrics_json)},
            {"mode": mode, "artifact": "prediction_csv", "path": str(result.prediction_csv)},
            {"mode": mode, "artifact": "grid_csv", "path": str(result.grid_csv)},
        ]
    )
    for label, attr in PLOT_SELECTIONS[mode]:
        plot_path = getattr(result, attr)
        artifact_rows.append({"mode": mode, "artifact": label, "path": str(plot_path) if plot_path else None})

pd.DataFrame(artifact_rows)


## 3. Inspect Metrics And CSV Outputs


In [ ]:
metric_rows = []
for mode, result in mfgp_results.items():
    metrics = json.loads(Path(result.metrics_json).read_text())
    metric_rows.append({"mode": mode, **metrics})

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)

for mode, result in mfgp_results.items():
    print(f"\n=== {mode} prediction CSV preview ===")
    display(pd.read_csv(result.prediction_csv).head())
    print(f"=== {mode} grid CSV preview ===")
    display(pd.read_csv(result.grid_csv).head())


## 4. Plot Display Helpers

The next four cells display each MF-GP experiment separately. Log parity plots and redundant log-sigma plots are intentionally omitted.


In [ ]:
def display_selected_mfgp_plots(mode: str) -> None:
    result = mfgp_results[mode]
    print(EXPERIMENT_TITLES[mode])
    for label, attr in PLOT_SELECTIONS[mode]:
        plot_path = getattr(result, attr)
        if plot_path is None:
            continue
        plot_path = Path(plot_path)
        if not plot_path.exists():
            print(f"[missing] {label}: {plot_path}")
            continue
        print(f"\n{label}")
        if plot_path.suffix.lower() == ".html":
            print(f"Open manually: {plot_path}")
        else:
            display(Image(filename=str(plot_path)))


## 5. Normal MF-GP

Baseline MF-GP using the original `y_cnp` and `y_raw` values.


In [ ]:
display_selected_mfgp_plots("linear")


## 6. Log HF: Emulate Log y

MF-GP trains the HF target as `log10(y_raw)` while keeping LF/CNP values in their original scale.


In [ ]:
display_selected_mfgp_plots("log_hf")


## 7. Log LF: Use Log y_cnp

MF-GP uses `log10(y_cnp)` for the LF/CNP side while keeping the HF target as original `y_raw`.


In [ ]:
display_selected_mfgp_plots("log_lf")


## 8. Log HF + Log LF

MF-GP uses `log10(y_raw)` for HF and `log10(y_cnp)` for LF/CNP.


In [ ]:
display_selected_mfgp_plots("log_both")


## 9. Optional: Inspect Per-Theta Validation Plots

If `theta_group_plot_dir` was generated, the next cell lists a few files from that directory.


In [ ]:
rows = []
for mode, result in mfgp_results.items():
    if result.theta_group_plot_dir is not None and Path(result.theta_group_plot_dir).exists():
        theta_plot_dir = Path(result.theta_group_plot_dir)
        theta_plots = sorted(theta_plot_dir.glob('*.png'))
        rows.append({"mode": mode, "theta_group_plot_count": len(theta_plots), "example": theta_plots[0].name if theta_plots else None})
    else:
        rows.append({"mode": mode, "theta_group_plot_count": 0, "example": None})

pd.DataFrame(rows)
